### Get user dialog system mapping

In [479]:
def get_user_agent_mapping(transcript_name):
    user_dialog_agent_mapping = {}
    agent_user_num_mapping = {}

    with open(transcript_name, "r") as transcript:
        for line in transcript:
            if "(POLICY:" in line:
                tmp = line.split()
                user = tmp[1].strip()
                policy = tmp[3].strip(")").strip()
                user_dialog_agent_mapping[user] = policy
                if policy not in agent_user_num_mapping:
                    agent_user_num_mapping[policy] = 0
                agent_user_num_mapping[policy] += 1
    return user_dialog_agent_mapping, agent_user_num_mapping

In [480]:
user_agent_mapping, ling_ad_agent_user_num = get_user_agent_mapping("combined/combined_transcript.txt")
tmp,  baseline_agent_user_num = get_user_agent_mapping("../mental_models/combined_data/transcript.txt")
user_agent_mapping |= tmp

### Expectations Met

In [482]:
import csv

key_mapping = {"fast answer": "fast answers",
               "clear answers": "clear answers/precise/detailed",
               "faster than expected": "fast answers",
               "Dialog too long": "Dialog too long/too many questions",
               "too many irrelevant questions": "Dialog too long/too many questions",
               "Can't ask follow-up questions": "Can't ask follow-up questions/ dialog too short",
               "accurate answers": "Accurate answers"}

ignore = set(['Needed keywords', 'Unclear Answers'])

ling_ad_expectation_labels = set()
user_expectations = {}
counts = {}
with open('combined/expectations_met.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        user_expectations[user] = []
        for entry in line[1:]:
            entry = entry.strip()
            if not entry == "" and not entry in ignore:
                if entry in key_mapping:
                    entry = key_mapping[entry]
                if entry not in counts:
                    counts[entry] = 0
                counts[entry] += 1
                if not "User" in user:
                    user_expectations[user].append(entry)
                ling_ad_expectation_labels.add(entry)

base_expectations_labels = set()
baseline_user_epectations = {}
baseline_counts = {}
with open('../mental_models/combined_data/expectations_met.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        baseline_user_epectations[user] = []
        for entry in line[1:]:
            entry = entry.strip()
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                if entry not in baseline_counts:
                    baseline_counts[entry] = 0
                baseline_counts[entry] += 1
                base_expectations_labels.add(entry)
                baseline_user_epectations[user].append(entry)



diff = ling_ad_expectation_labels.difference(base_expectations_labels)
for key in diff:
    if key not in key_mapping:
        print(key)


print("BASELINE")
print(sorted(baseline_counts.items()))
print("LING AD")
print(sorted(counts.items()))

BASELINE
[('Accurate answers', 7), ("Can't ask follow-up questions/ dialog too short", 3), ('Dialog too fast/short', 9), ('Dialog too long/too many questions', 3), ("Didn't understand/couldn't answer", 12), ('Expectations Met/Exceeded', 26), ('Expectations not met', 14), ('Good at follow-up questions', 3), ('Negative Expectations Met', 8), ('Robotic', 7), ('Struggled with complex input', 10), ('clear answers/precise/detailed', 2), ('fast answers', 6), ('general answers', 13)]
LING AD
[('Accurate answers', 12), ("Can't ask follow-up questions/ dialog too short", 5), ('Dialog too long/too many questions', 4), ("Didn't understand/couldn't answer", 18), ('Expectations Met/Exceeded', 31), ('Expectations not met', 14), ('Good at follow-up questions', 1), ('Negative Expectations Met', 3), ('Robotic', 5), ('Struggled with complex input', 1), ('clear answers/precise/detailed', 12), ('fast answers', 8), ('general answers', 9)]


In [325]:
def get_exp_counts_by_policy(user_exps, user_agent_mapping):
    exps_counts = {}
    for user in user_exps:
        if user not in user_agent_mapping:
            print(user)
            continue
        policy = user_agent_mapping[user]
        if policy not in exps_counts:
            exps_counts[policy] = {}
        for exp in user_exps[user]:
            if exp not in exps_counts[policy]:
                exps_counts[policy][exp] = 0
            exps_counts[policy][exp] += 1
    return exps_counts

In [326]:
baseline_exps = get_exp_counts_by_policy(baseline_user_epectations, user_agent_mapping)
ling_ad_exps = get_exp_counts_by_policy(user_expectations, user_agent_mapping)

User


In [327]:
for policy in baseline_exps:
    print(f"POLICY: {policy}")
    print("baseline")
    print(sorted(baseline_exps[policy].items()))
    print("ling ad")
    print(sorted(ling_ad_exps[policy].items()))


POLICY: cts
baseline
[('Accurate answers', 3), ('Dialog too long/too many questions', 1), ('Expectations Met/Exceeded', 12), ('Expectations not met', 3), ('Good at follow-up questions', 3), ('Robotic', 4), ('Struggled with complex input', 5), ('clear answers/precise/detailed', 2), ('fast answers', 2), ('general answers', 3)]
ling ad
[('Accurate answers', 5), ("Can't ask follow-up questions/ dialog too short", 2), ("Didn't understand/couldn't answer", 4), ('Expectations Met/Exceeded', 9), ('Expectations not met', 3), ('Negative Expectations Met', 2), ('Robotic', 4), ('clear answers/precise/detailed', 4), ('fast answers', 2), ('general answers', 1)]
POLICY: hdc
baseline
[('Accurate answers', 1), ('Dialog too fast/short', 2), ('Dialog too long/too many questions', 2), ("Didn't understand/couldn't answer", 8), ('Expectations Met/Exceeded', 6), ('Expectations not met', 8), ('Negative Expectations Met', 5), ('Struggled with complex input', 3), ('fast answers', 2), ('general answers', 3)]
lin

In [328]:
from seaborn import barplot
import plotly.graph_objects as go
import plotly.express as px

import pandas as pd
expectations_data = []
simple_labels = ["general answers"]
complex_labels = {"Expectations not met": ['Expectations not met', 'Negative Expectations Met'],
                  "High quality answers": ["Accurate answers", "clear answers/precise/detailed"],
                  "Dialog too short": ["Dialog too fast/short"],
                  "Expectations met/exceeded": ["Expectations Met/Exceeded"]}
base_data  = {}
ling_ad_data = {}
for policy in baseline_exps:
    base_data[policy] = {}
    ling_ad_data[policy] = {}
    for label in simple_labels:
        expectations_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": baseline_exps[policy][label]
        })
        base_data[policy][label] = baseline_exps[policy][label]
        expectations_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": ling_ad_exps[policy][label]
        })
        ling_ad_data[policy][label] = ling_ad_exps[policy][label]
    for label in complex_labels:
        expectations_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": sum([baseline_exps[policy][l] if l in baseline_exps[policy] else 0 for l in complex_labels[label]])
        })
        base_data[policy][label] = sum([baseline_exps[policy][l] if l in baseline_exps[policy] else 0 for l in complex_labels[label]])
        expectations_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": sum([ling_ad_exps[policy][l] if l in ling_ad_exps[policy] else 0 for l in complex_labels[label]])
        })
        ling_ad_data[policy][label] = sum([ling_ad_exps[policy][l] if l in ling_ad_exps[policy] else 0 for l in complex_labels[label]])
expectations_data = pd.DataFrame(expectations_data)


fig = px.bar(expectations_data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.show()

In [329]:
from scipy.stats import barnard_exact

def calc_stats(base_code_counts, ling_ad_code_counts, baseline_agent_user_num, ling_ad_agent_user_num, simple_labels, complex_labels):
    for policy in ['cts', 'hdc', 'faq']:
        keys = set(simple_labels + [key for key in complex_labels])
        print(policy)
        for key in keys:
            print(key)
            if key in simple_labels:
                baseline_yeses = base_code_counts[policy][key] if key in base_code_counts[policy] else 0
                ling_ad_yeses = ling_ad_code_counts[policy][key] if key in ling_ad_code_counts[policy] else 0

            elif key in complex_labels:
                baseline_yeses = 0
                ling_ad_yeses = 0
                for label in complex_labels[key]:
                    baseline_yeses += base_code_counts[policy][label] if label in base_code_counts[policy] else 0
                    ling_ad_yeses  += ling_ad_code_counts[policy][label] if label in ling_ad_code_counts[policy] else 0
            baseline_nos = baseline_agent_user_num[policy] - baseline_yeses
            ling_ad_nos = ling_ad_agent_user_num[policy] - ling_ad_yeses
            print(barnard_exact([[baseline_yeses, ling_ad_yeses], [baseline_nos, ling_ad_nos]]))


In [330]:
def calc_stats_combined(base_code_counts, ling_ad_code_counts, baseline_agent_user_num, ling_ad_agent_user_num, simple_labels, complex_labels):
    keys = set(simple_labels + [key for key in complex_labels])
    for key in keys:
        print(key)
        baseline_yeses = 0
        ling_ad_yeses = 0
        baseline_nos = 0
        ling_ad_nos = 0
        for policy in ['cts', 'hdc', 'faq']:            
            if key in simple_labels:
                baseline_yeses += base_code_counts[policy][key] if key in base_code_counts[policy] else 0
                ling_ad_yeses += ling_ad_code_counts[policy][key] if key in ling_ad_code_counts[policy] else 0

            elif key in complex_labels:
                for label in complex_labels[key]:
                    baseline_yeses += base_code_counts[policy][label] if label in base_code_counts[policy] else 0
                    ling_ad_yeses  += ling_ad_code_counts[policy][label] if label in ling_ad_code_counts[policy] else 0
            baseline_nos += baseline_agent_user_num[policy] - baseline_yeses
            ling_ad_nos += ling_ad_agent_user_num[policy] - ling_ad_yeses
        print(barnard_exact([[baseline_yeses, ling_ad_yeses], [baseline_nos, ling_ad_nos]]))

In [331]:
simple_labels = ["general answers"]
complex_labels = {"Expectations not met": ['Expectations not met', 'Negative Expectations Met'],
                  "High quality answers": ["Accurate answers", "clear answers/precise/detailed"],
                  "Dialog too short": ["Dialog too fast/short"],
                  "Expectations met/exceeding": ["Expectations Met/Exceeded"]}

calc_stats(base_code_counts=baseline_exps,
           ling_ad_code_counts=ling_ad_exps,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

print("============================================")
calc_stats_combined(base_code_counts=baseline_exps,
           ling_ad_code_counts=ling_ad_exps,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

cts
general answers
BarnardExactResult(statistic=1.0654883777715443, pvalue=0.31973643658655)
High quality answers
BarnardExactResult(statistic=-1.0394654940159311, pvalue=0.31988848702377354)
Expectations not met
BarnardExactResult(statistic=-0.6609074099624134, pvalue=0.5480635246398544)
Dialog too short
BarnardExactResult(statistic=0.0, pvalue=1.0)
Expectations met/exceeding
BarnardExactResult(statistic=0.8385286386076907, pvalue=0.5300505615159518)
hdc
general answers
BarnardExactResult(statistic=-0.798241029395918, pvalue=0.530145111147903)
High quality answers
BarnardExactResult(statistic=-1.7311539583127296, pvalue=0.09029672323859271)
Expectations not met
BarnardExactResult(statistic=1.6297456741867096, pvalue=0.11269292039253756)
Dialog too short
BarnardExactResult(statistic=1.3925355768726773, pvalue=0.22320979137091504)
Expectations met/exceeding
BarnardExactResult(statistic=-0.9200433880820621, pvalue=0.5288269670122522)
faq
general answers
BarnardExactResult(statistic=1.80

#### Notes:
- Important codes:
    - Expectations not met + Negative Expectations Met (merge)
    - Expectations met
    - Accurate Answers + clear answers/precise/detailed (merge)
    - General Answers
    - Dialog too short

#### Expectations for different dialog systems:
- **CTS**: 
    - Although the expectations were in general higher, (fewer being met), users were more satisfied with the answer, and found it better suited to their questions
- **HDC**:
    - More users thought the answers were clear, but users struggled to get through general information to find an answer
    - Less effect of linguistic adaptation
    - Users often broke dialog off early
- **FAQ**:
    - Dialogs were seen as much more appropriate in length
    - Expectations were much better met
    - Answers were seen as better in quality and more personalized

### System Strengths

In [483]:
key_mapping = {"Clear/concise answers": "Clear/concise/detailed answers",
               "Detailed answers": "Clear/concise/detailed answers",
               "Clear/concise/detailed answers/": "Clear/concise/detailed answers"}

ignore = set(['sample answers'])

user_strengths = {}
ling_ad_strengths_labels = set()
with open('combined/system_strengths.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        user_strengths[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                user_strengths[user].append(entry)
                ling_ad_strengths_labels.add(entry)

baseline_user_strengths = {}
base_strengths_labels = set()
with open('../mental_models/combined_data/system_strengths.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        baseline_user_strengths[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                baseline_user_strengths[user].append(entry)
                base_strengths_labels.add(entry)

diff = base_strengths_labels.symmetric_difference(ling_ad_strengths_labels)
for key in diff:
    if key not in key_mapping:
        print(key)

sample answers
Good Communication


In [333]:
baseline_strengths = get_exp_counts_by_policy(baseline_user_strengths, user_agent_mapping)
ling_ad_strengths = get_exp_counts_by_policy(user_strengths, user_agent_mapping)

for policy in baseline_strengths:
    print(f"POLICY: {policy}")
    print("baseline")
    print(sorted(baseline_strengths[policy].items()))
    print("ling ad")
    print(sorted(ling_ad_strengths[policy].items()))

User
POLICY: cts
baseline
[('Accurate answers', 4), ('Answer common questions', 5), ('Answer specific questions', 1), ('Clear/concise/detailed answers', 2), ('Could ask good follow-up questions', 3), ('Fast', 5), ('Provided sample answers', 1), ('Stay on topic', 2), ('Understand keywords', 3), ('Understand my question', 1)]
ling ad
[('Accurate answers', 3), ('Answer common questions', 1), ('Answer specific questions', 3), ('Clear/concise/detailed answers', 6), ('Fast', 2), ('Good Communication', 1), ('Nothing', 1), ('Provided sample answers', 1), ('Stay on topic', 1), ('Understand keywords', 2), ('Understand my question', 2)]
POLICY: hdc
baseline
[('Accurate answers', 2), ('Answer common questions', 6), ('Answer specific questions', 1), ('Clear/concise/detailed answers', 1), ('Fast', 6), ('Further reading suggestions', 3), ('Nothing', 4), ('Understand keywords', 1), ('sample answers', 3)]
ling ad
[('Accurate answers', 3), ('Answer common questions', 4), ('Answer specific questions', 3)

In [334]:
strengths_data = []
simple_labels = ["Good Communication"]
complex_labels = {"High quality answers": ["Accurate answers", "Clear/concise/detailed answers"]}

for policy in baseline_strengths:
    for label in simple_labels:
        strengths_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": baseline_strengths[policy][label] if label in baseline_strengths[policy] else 0
        })
        strengths_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": ling_ad_strengths[policy][label] if label in ling_ad_strengths[policy] else 0
        })
    for label in complex_labels:
        strengths_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": sum([baseline_strengths[policy][l] if l in baseline_strengths[policy] else 0 for l in complex_labels[label]])
        })
        strengths_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": sum([ling_ad_strengths[policy][l] if l in ling_ad_strengths[policy] else 0 for l in complex_labels[label]])
        })
strengths_data = pd.DataFrame(strengths_data)

strengths_data = pd.concat([strengths_data[(strengths_data.code == 'Good Communication')], expectations_data[(expectations_data.code == 'High quality answers')]])


fig = px.bar(strengths_data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.show()

In [335]:
simple_labels = ["Good Communication"]
complex_labels = {"High quality answers": ["Accurate answers", "Clear/concise/detailed answers"]}
calc_stats(base_code_counts=baseline_strengths,
           ling_ad_code_counts=ling_ad_strengths,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

print("========================================")
calc_stats_combined(base_code_counts=baseline_strengths,
           ling_ad_code_counts=ling_ad_strengths,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

cts
High quality answers
BarnardExactResult(statistic=-0.7268454466244446, pvalue=0.4986348804000689)
Good Communication
BarnardExactResult(statistic=-0.9802099223816721, pvalue=0.5146519278296747)
hdc
High quality answers
BarnardExactResult(statistic=-0.798241029395918, pvalue=0.530145111147903)
Good Communication
BarnardExactResult(statistic=-1.0275230272011286, pvalue=0.3643351470354715)
faq
High quality answers
BarnardExactResult(statistic=-1.1277732706875612, pvalue=0.28347294933926637)
Good Communication
BarnardExactResult(statistic=-0.9805184146237956, pvalue=0.5146678668024798)
High quality answers
BarnardExactResult(statistic=-1.6965773480170028, pvalue=0.09187854085397387)
Good Communication
BarnardExactResult(statistic=-1.7389652095427326, pvalue=0.08949991917484398)


In [336]:
# Merge high quality answer feedback
users = set()
for user in user_expectations:
    if "Accurate answers" in user_expectations[user] or "Clear/concise/detailed answers" in user_expectations[user]:
        users.add(user)
for user in user_strengths:
    if "Accurate answers" in user_strengths[user] or "Clear/concise/detailed answers" in user_strengths[user]:
        users.add(user)

high_quality_per_policy = {}
for user in users:
    if user == "\ufeffUser":
        continue
    policy = user_agent_mapping[user]
    if policy not in high_quality_per_policy:
        high_quality_per_policy[policy] = 0
    high_quality_per_policy[policy] += 1

baseline_users = set()
for user in baseline_user_epectations:
    if "Accurate answers" in baseline_user_epectations[user] or "Clear/concise/detailed answers" in baseline_user_epectations[user]:
        baseline_users.add(user)
for user in baseline_user_strengths:
    if "Accurate answers" in baseline_user_strengths[user] or "Clear/concise/detailed answers" in baseline_user_strengths[user]:
        baseline_users.add(user)

baseline_high_quality_per_policy = {}
for user in baseline_users:
    policy = user_agent_mapping[user]
    if policy not in baseline_high_quality_per_policy:
        baseline_high_quality_per_policy[policy] = 0
    baseline_high_quality_per_policy[policy] += 1

print(len(baseline_users))
print(baseline_high_quality_per_policy)
print(len(users))
print(high_quality_per_policy)

14
{'faq': 6, 'cts': 6, 'hdc': 2}
26
{'faq': 8, 'hdc': 6, 'cts': 11}


#### Notes:
**Important labels**
- Accurate Answers + clear/concise/detailed answers
- Good communication

#### Analsys per condition
- **CTS**:
    - Users were more satisfied with the answers

### System Weaknesses

In [484]:
key_mapping = {"understanding my question": "understanding/answering my question",
               "Interpret long/complex questions": "Interpret long/complex/vauge questions"}

ling_ad_weaknesses_labels = set()
user_weaknesses = {}
with open('combined/system_weaknesses.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        user_weaknesses[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                user_weaknesses[user].append(entry)
                ling_ad_weaknesses_labels.add(entry)

base_weaknesses_labels = set()
baseline_user_weaknesses = {}
with open('../mental_models/combined_data/system_weaknesses.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        baseline_user_weaknesses[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                baseline_user_weaknesses[user].append(entry)
                base_weaknesses_labels.add(entry)

diff = base_weaknesses_labels.symmetric_difference(ling_ad_weaknesses_labels)
for key in diff:
    if key not in key_mapping:
        print(key)

In [338]:
baseline_weaknesses = get_exp_counts_by_policy(baseline_user_weaknesses, user_agent_mapping)
ling_ad_weaknesses = get_exp_counts_by_policy(user_weaknesses, user_agent_mapping)

for policy in baseline_weaknesses:
    print(f"POLICY: {policy}")
    print("baseline")
    print(sorted(baseline_weaknesses[policy].items()))
    print("ling ad")
    print(sorted(ling_ad_weaknesses[policy].items()))

User
POLICY: cts
baseline
[('Ask relevant questions', 2), ('Be friendly/personable', 2), ('Give very specific answers', 2), ('Interpret long/complex/vauge questions', 10), ('Summarize information', 1), ("Tell if it didn't know the answer", 1)]
ling ad
[('Ask relevant questions', 1), ('Give very specific answers', 4), ('Interpret long/complex/vauge questions', 5), ('Nothing', 6), ('Provide enough context/detail', 2), ('Summarize information', 2), ('dialog too long', 1), ('follow up questions', 1), ('understanding/answering my question', 4)]
POLICY: hdc
baseline
[('Be friendly/personable', 1), ('Give very specific answers', 7), ('Only understand keywords', 2), ('Provide enough context/detail', 1), ('dialog too long', 2), ('follow up questions', 1), ('understanding/answering my question', 8)]
ling ad
[('Give very specific answers', 2), ('Interpret long/complex/vauge questions', 6), ('Nothing', 4), ('dialog too long', 2), ('follow up questions', 1), ('understanding/answering my question', 

In [339]:
weaknesses_data = []
simple_labels = []
complex_labels = {"Unfriendly": ["Be friendly/personable"],
                  "No specific answers": ["Give very specific answers"]}

for policy in baseline_weaknesses:
    for label in simple_labels:
        weaknesses_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": baseline_weaknesses[policy][label] if label in baseline_weaknesses[policy] else 0
        })
        weaknesses_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": ling_ad_weaknesses[policy][label] if label in ling_ad_weaknesses[policy] else 0
        })
    for label in complex_labels:
        weaknesses_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": sum([baseline_weaknesses[policy][l] if l in baseline_weaknesses[policy] else 0 for l in complex_labels[label]])
        })
        weaknesses_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": sum([ling_ad_weaknesses[policy][l] if l in ling_ad_weaknesses[policy] else 0 for l in complex_labels[label]])
        })
weaknesses_data = pd.DataFrame(weaknesses_data)

data = pd.concat([strengths_data, weaknesses_data, expectations_data[expectations_data["code"] == "Expectations met/exceeded"], expectations_data[expectations_data["code"] == "Expectations not met"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [340]:
simple_labels = []
complex_labels = {"Unfriendly": ["Be friendly/personable"],
                  "No specific answers": ["Give very specific answers"]}
calc_stats(base_code_counts=baseline_weaknesses,
           ling_ad_code_counts=ling_ad_weaknesses,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

cts
No specific answers
BarnardExactResult(statistic=-0.7767995858307438, pvalue=0.5316240692732596)
Unfriendly
BarnardExactResult(statistic=1.4603014555895144, pvalue=0.1521931315017667)
hdc
No specific answers
BarnardExactResult(statistic=1.6561482664082587, pvalue=0.1087056509475289)
Unfriendly
BarnardExactResult(statistic=0.9808174350556228, pvalue=0.5146833986271837)
faq
No specific answers
BarnardExactResult(statistic=3.1867129491763664, pvalue=0.001401442223463608)
Unfriendly
BarnardExactResult(statistic=0.6258738121720493, pvalue=0.5986268380018602)


#### Notes:
**Important labels**
- Be friendly/personable
- Give specific answers

#### Analsys per condition
- Personal in this case refers also to whether the content is personalized for each user
- CTS users had higher expectations 

### Get counts of all mental models for users

In [423]:
def collect_user_mms(expectations, strengths, weaknesses):

    user_mms = {}
    expectations_key_map = {"Negative Expectations Met": "Expectations not met",
                            "Expectations Met/Exceeded": "Expectations met",
                            "Dialog too long/too many questions": "Interaction too long",
                            "Can't ask follow-up questions/ dialog too short": "Interaction too short",
                            "Expectations not met": "Expectations not met",
                            "general answers": "General answers",
                            "Didn't understand/couldn't answer": "Could not understand/answer",
                            "Accurate answers": "High quality answers",
                            "clear answers/precise/detailed": "High quality answers"}

    strenghts_key_map = {"Fast": "Fast Interaction",
                        "Clear/concise/detailed answers": "High quality answers",
                        "Answer common questions": "Could answer common questions",
                        "Answer specific questions": "Could answer difficult questions",
                        "Good Communication": "Good communication",
                        "Accurate answers": "High quality answers",
                        "Understand keywords": "Understood Keywords"}

    weaknesses_key_map = {"understanding/answering my question": "Could not understand/answer",
                        "Nothing": "Nothing",
                        "Give very specific answers": "No specific answers",
                        "Only understand keywords": "Understood Keywords",
                        "Interpret long/complex/vauge questions": "Could not understand/answer",
                        "dialog too long": "Interaction too long",
                        "follow up questions": "Interaction too short",
                        "Be friendly/personable": "Unfriendly"}


    all_keys = set()
    for user in expectations:

        if user != '\ufeffUser':
            user_mms[user] = set()
            for key in expectations[user]:
                if key in expectations_key_map:
                    user_mms[user].add(expectations_key_map[key])
                    all_keys.add(expectations_key_map[key])

            for key in strengths[user]:
                if key in strenghts_key_map:
                    user_mms[user].add(strenghts_key_map[key])
                    all_keys.add(strenghts_key_map[key])

            for key in weaknesses[user]:
                if key in weaknesses_key_map:
                    user_mms[user].add(weaknesses_key_map[key])
                    all_keys.add(weaknesses_key_map[key])


    return user_mms, all_keys


exp_user_mms, all_keys = collect_user_mms(user_expectations, user_strengths, user_weaknesses)
baseline_user_mms, all_keys2 = collect_user_mms(baseline_user_epectations, baseline_user_strengths, baseline_user_weaknesses)
all_keys.update(all_keys2)

In [368]:
import csv

all_keys = sorted(list(all_keys))

mms_data = []

with open("user_mms.tsv", "w") as outfile:
    writer = csv.writer(outfile, delimiter='\t', quotechar='|', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["username", "agent", "style"] + [key for key in all_keys])
    for user in baseline_user_mms:
        writer.writerow([user, user_agent_mapping[user], "baseline"] + [1 if key in baseline_user_mms[user] else 0 for key in all_keys])
    for user in exp_user_mms:
        writer.writerow([user, user_agent_mapping[user], "ling_ad"] + [1 if key in exp_user_mms[user] else 0 for key in all_keys])


In [409]:
def get_mms_per_policy(user_mms, user_agent_mapping):
    mms_per_policy = {}
    for user in user_mms:
        policy = user_agent_mapping[user]
        if policy not in mms_per_policy:
            mms_per_policy[policy] = {}
        for key in user_mms[user]:
            if key not in mms_per_policy[policy]:
                mms_per_policy[policy][key] = 0
            mms_per_policy[policy][key] += 1
    return mms_per_policy


In [424]:
baseline_mms_per_policy = get_exp_counts_by_policy(baseline_user_mms, user_agent_mapping)
exp_mms_per_policy = get_exp_counts_by_policy(exp_user_mms, user_agent_mapping)
print(exp_mms_per_policy)
print(baseline_mms_per_policy)

{'cts': {'Could not understand/answer': 11, 'Could answer difficult questions': 3, 'High quality answers': 11, 'Interaction too short': 3, 'Good communication': 1, 'Expectations met': 9, 'No specific answers': 4, 'Expectations not met': 5, 'Understood Keywords': 2, 'Fast Interaction': 2, 'Interaction too long': 1, 'Nothing': 6, 'General answers': 1, 'Could answer common questions': 1}, 'faq': {'Could answer common questions': 1, 'Could not understand/answer': 10, 'Expectations met': 12, 'High quality answers': 9, 'Understood Keywords': 2, 'General answers': 2, 'Could answer difficult questions': 3, 'Fast Interaction': 7, 'Expectations not met': 4, 'Interaction too short': 2, 'Unfriendly': 1, 'Nothing': 5, 'Good communication': 1, 'Interaction too long': 1}, 'hdc': {'Interaction too long': 4, 'High quality answers': 7, 'Expectations met': 9, 'Could answer common questions': 4, 'General answers': 5, 'Could not understand/answer': 10, 'No specific answers': 2, 'Nothing': 4, 'Could answer 

In [425]:
mental_models_data = []
for policy in baseline_mms_per_policy:
    for key in all_keys:
        mental_models_data.append({
            "code": key,
            "policy": policy,
            "condition": "Baseline",
            "count": baseline_mms_per_policy[policy][key] if key in baseline_mms_per_policy[policy] else 0
        })
        mental_models_data.append({
            "code": key,
            "policy": policy,
            "condition": "Ling. Ad.",
            "count": exp_mms_per_policy[policy][key] if key in exp_mms_per_policy[policy] else 0
        })

mental_models_data = pd.DataFrame(mental_models_data)

In [398]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "Good communication"],
                 mental_models_data[mental_models_data["code"] == "High quality answers"],
                 mental_models_data[mental_models_data["code"] == "Unfriendly"],
                 mental_models_data[mental_models_data["code"] == "No specific answers"],
                 mental_models_data[mental_models_data["code"] == "Expectations met"],
                 mental_models_data[mental_models_data["code"] == "Expectations not met"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [405]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "Expectations met"],
                 mental_models_data[mental_models_data["code"] == "Expectations not met"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [406]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "High quality answers"],
                 mental_models_data[mental_models_data["code"] == "General answers"],
                 mental_models_data[mental_models_data["code"] == "No specific answers"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [412]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "Fast Interaction"],
                 mental_models_data[mental_models_data["code"] == "Interaction too short"],
                 mental_models_data[mental_models_data["code"] == "Interaction too long"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [413]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "Good communication"],
                 mental_models_data[mental_models_data["code"] == "Unfriendly"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [428]:
data = pd.concat([mental_models_data[mental_models_data["code"] == "Could not understand/answer"],
                 mental_models_data[mental_models_data["code"] == "Could answer common questions"],
                 mental_models_data[mental_models_data["code"] == "Understood Keywords"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

# User Experience

### User Likes

In [485]:
key_mapping = {"Robot/don't need person": "Robot/don't need a person"}

ling_ad_likes_labels = set()
user_likes = {}
with open('combined/user_likes.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        user_likes[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                user_likes[user].append(entry)
                ling_ad_likes_labels.add(entry)

base_likes_labels = set()
baseline_user_likes = {}
with open('../mental_models/combined_data/user_likes.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        baseline_user_likes[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                baseline_user_likes[user].append(entry)
                base_likes_labels.add(entry)

diff = base_likes_labels.symmetric_difference(ling_ad_likes_labels)
for key in diff:
    if key not in key_mapping:
        print(key)

Friendly language style
Easy to understand


In [486]:
baseline_likes = get_exp_counts_by_policy(baseline_user_likes, user_agent_mapping)
ling_ad_likes = get_exp_counts_by_policy(user_likes, user_agent_mapping)

for policy in baseline_likes:
    print(f"POLICY: {policy}")
    print("baseline")
    print(sorted(baseline_likes[policy].items()))
    print("ling ad")
    print(sorted(ling_ad_likes[policy].items()))

User
POLICY: cts
baseline
[('Basic questions', 1), ("Don't like chatbots", 1), ('Easy to use', 5), ('Fast', 15), ('Straight to the point', 3), ('ask good follow-up questions', 1)]
ling ad
[('Easy to understand', 2), ('Easy to use', 4), ('Fast', 11), ('Friendly language style', 3), ('Straight to the point', 3), ('accurate', 3)]
POLICY: hdc
baseline
[('Basic questions', 1), ('Easy to use', 5), ('Fast', 7), ('Links to further reading', 1), ('Nothing', 5), ("Robot/don't need a person", 1), ('Straight to the point', 2), ('accurate', 1), ('polite/professional', 1), ('sample answers', 1)]
ling ad
[('Basic questions', 1), ('Easy to use', 2), ('Fast', 10), ('Friendly language style', 1), ('Nothing', 3), ("Robot/don't need a person", 2), ('Straight to the point', 2), ('accurate', 2), ('sample answers', 2)]
POLICY: faq
baseline
[('Easy to use', 7), ('Fast', 12), ('Links to further reading', 2), ('Nothing', 2), ('Straight to the point', 2), ('accurate', 1), ('polite/professional', 1)]
ling ad
[('E

In [286]:
likes_data = []
simple_labels = []
complex_labels = {"Liked language style": ["Friendly language style", "Robot/don't need a person", "polite/professional"],
                  "Easy to understand": ["Easy to understand", "Straight to the point"]}

for policy in baseline_likes:
    for label in simple_labels:
        likes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": baseline_likes[policy][label] if label in baseline_likes[policy] else 0
        })
        likes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": ling_ad_likes[policy][label] if label in ling_ad_likes[policy] else 0
        })
    for label in complex_labels:
        likes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": sum([baseline_likes[policy][l] if l in baseline_likes[policy] else 0 for l in complex_labels[label]])
        })
        likes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": sum([ling_ad_likes[policy][l] if l in ling_ad_likes[policy] else 0 for l in complex_labels[label]])
        })
likes_data = pd.DataFrame(likes_data)


fig = px.bar(likes_data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.show()

In [305]:
simple_labels = []
complex_labels = {"Liked language style": ["Friendly language style", "Robot/don't need a person", "polite/professional"],
                  "Easy to understand": ["Easy to understand", "Straight to the point"]}

calc_stats(base_code_counts=baseline_likes,
           ling_ad_code_counts=ling_ad_likes,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

print("========================================")

calc_stats_combined(base_code_counts=baseline_likes,
           ling_ad_code_counts=ling_ad_likes,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

cts
Liked language style
BarnardExactResult(statistic=-1.711632992203644, pvalue=0.10434954447343456)
Easy to understand
BarnardExactResult(statistic=-0.6609074099624134, pvalue=0.5480635246398544)
hdc
Liked language style
BarnardExactResult(statistic=-0.5093182806634906, pvalue=0.7101729549249178)
Easy to understand
BarnardExactResult(statistic=-0.047262736206933956, pvalue=0.999992978797879)
faq
Liked language style
BarnardExactResult(statistic=-0.5430375723257488, pvalue=0.6818928026311503)
Easy to understand
BarnardExactResult(statistic=-0.7774331127958555, pvalue=0.5316442347914527)
Liked language style
BarnardExactResult(statistic=-1.5669708809289402, pvalue=0.12910755767373736)
Easy to understand
BarnardExactResult(statistic=-0.9792918165749844, pvalue=0.5294710812776148)


### User Dislikes

In [487]:
key_mapping = {"Too long": "Conversations were too long/hard to get answer",
               "Conversations were too long": "Conversations were too long/hard to get answer"}

ignore = set(['short answers', "Short answers"])

ling_ad_dislikes_labels = set()
user_dislikes = {}
with open('combined/user_dislikes.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        user_dislikes[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                user_dislikes[user].append(entry)
                ling_ad_dislikes_labels.add(entry)

base_dislikes_labels = set()
baseline_user_dislikes = {}
with open('../mental_models/combined_data/user_dislikes.csv', 'r') as infile:
    reader = csv.reader(infile, delimiter=";")
    for line in reader:
        user = line[0]
        baseline_user_dislikes[user] = []
        for entry in line[1:]:
            if not entry == "":
                if entry in key_mapping:
                    entry = key_mapping[entry]
                baseline_user_dislikes[user].append(entry)
                base_dislikes_labels.add(entry)

diff = base_dislikes_labels.symmetric_difference(ling_ad_dislikes_labels)
for key in diff:
    if key not in key_mapping:
        print(key)

Short answers


In [82]:
baseline_dislikes = get_exp_counts_by_policy(baseline_user_dislikes, user_agent_mapping)
ling_ad_dislikes = get_exp_counts_by_policy(user_dislikes, user_agent_mapping)

for policy in baseline_dislikes:
    print(f"POLICY: {policy}")
    print("baseline")
    print(sorted(baseline_dislikes[policy].items()))
    print("ling ad")
    print(sorted(ling_ad_dislikes[policy].items()))

User
POLICY: cts
baseline
[("Can't handle complex input", 8), ('Conversations were too long/hard to get answer', 3), ("Couldn't understand my question", 1), ('Fixed/impersonal text', 3), ('Giving private information to bots', 1), ('Needing to type', 1), ('No direction to further resources', 3), ('Nothing', 2), ('Outdated content', 1)]
ling ad
[("Can't handle complex input", 3), ('Conversations were too long/hard to get answer', 6), ("Couldn't understand my question", 4), ('Fixed/impersonal text', 3), ('No direction to further resources', 2), ('No-follow up questions', 1), ('Nothing', 6)]
POLICY: hdc
baseline
[('Conversations were too long/hard to get answer', 4), ("Couldn't understand my question", 8), ('Everything', 4), ('Fixed/impersonal text', 3), ('Nothing', 1), ('Outdated content', 1)]
ling ad
[('Conversations were too long/hard to get answer', 4), ("Couldn't understand my question", 6), ('Everything', 1), ('Fixed/impersonal text', 4), ('Needing to type', 1), ('Nothing', 5), ('Sho

In [292]:
disklikes_data = []
simple_labels = ["Fixed/impersonal text", "Everything", "Nothing"]
complex_labels = {}
# complex_labels = {"Dialog too long": ["Conversations were too long/hard to get answer"]}

for policy in baseline_dislikes:
    for label in complex_labels:
        disklikes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": sum([baseline_dislikes[policy][l] if l in baseline_dislikes[policy] else 0 for l in complex_labels[label]])
        })
        disklikes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": sum([ling_ad_dislikes[policy][l] if l in ling_ad_dislikes[policy] else 0 for l in complex_labels[label]])
        })    
    for label in simple_labels:
        disklikes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Baseline",
            "count": baseline_dislikes[policy][label] if label in baseline_dislikes[policy] else 0
        })
        disklikes_data.append({
            "code": label,
            "policy": policy,
            "condition": f"Ling. Ad.",
            "count": ling_ad_dislikes[policy][label] if label in ling_ad_dislikes[policy] else 0
        })
disklikes_data = pd.DataFrame(disklikes_data)
data = pd.concat([likes_data, disklikes_data])


fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01,
    font=dict(size=15)
),
font=dict(size=18))
fig.show()

In [306]:
simple_labels = ["Fixed/impersonal text", "Everything", "Nothing"]
complex_labels = {"Dialog too long": ["Conversations were too long/hard to get answer"]}
calc_stats(base_code_counts=baseline_dislikes,
           ling_ad_code_counts=ling_ad_dislikes,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

print("=======================================")

calc_stats_combined(base_code_counts=baseline_dislikes,
           ling_ad_code_counts=ling_ad_dislikes,
           baseline_agent_user_num=baseline_agent_user_num,
           ling_ad_agent_user_num=ling_ad_agent_user_num,
           simple_labels=simple_labels,
           complex_labels=complex_labels)

cts
Fixed/impersonal text
BarnardExactResult(statistic=0.060268933383419765, pvalue=0.9999191102585605)
Nothing
BarnardExactResult(statistic=-1.3919996776199501, pvalue=0.18652387934012918)
Dialog too long
BarnardExactResult(statistic=-0.9636051245534001, pvalue=0.5287979239147912)
Everything
BarnardExactResult(statistic=0.0, pvalue=1.0)
hdc
Fixed/impersonal text
BarnardExactResult(statistic=-0.4520484044528515, pvalue=0.7180873487553778)
Nothing
BarnardExactResult(statistic=-1.7311539583127296, pvalue=0.09029672323859271)
Dialog too long
BarnardExactResult(statistic=-0.06793540675709929, pvalue=0.9997011934541375)
Everything
BarnardExactResult(statistic=1.3157388917140176, pvalue=0.24326267090615292)
faq
Fixed/impersonal text
BarnardExactResult(statistic=2.7868981099546533, pvalue=0.0053575687779490455)
Nothing
BarnardExactResult(statistic=-1.6560012590591715, pvalue=0.10986619118720713)
Dialog too long
BarnardExactResult(statistic=-0.9805184146237956, pvalue=0.5146678668024798)
Every

In [489]:
def collect_user_usability(likes, dislikes):

    user_ratings = {}

    likes_key_mapping = {"polite/professional": "Liked language style",
                         "Straight to the point": "Easy to understand",
                         "Fast": "Fast",
                         "Easy to use": "Easy to use",
                         "Easy to understand": "Easy to understand",
                         "Friendly language style": "Liked language style",
                         "accurate": "Accurate",
                         "Robot/don't need a person": "Liked language style"}

    dislikes_key_mapping = {"Nothing": "disliked_nothing",
                            "Short answers": "Interaction too short",
                            "No-follow up questions": "Interaction too short",
                            "Can't handle complex input": "Could not understand/answer",
                            "Conversations were too long/hard to get answer": "Interaction too long",
                            "Everything": "Disliked everything",
                            "Fixed/impersonal text": "Impersonal/Fixed text",
                            "Couldn't understand my question": "Could not understand/answer"}

    all_keys = set()
    for user in likes:
        if user != '\ufeffUser':
            user_ratings[user] = set()
            for key in likes[user]:
                if key in likes_key_mapping:
                    user_ratings[user].add(likes_key_mapping[key])
                    all_keys.add(likes_key_mapping[key])

            for key in dislikes[user]:
                if key in dislikes_key_mapping:
                    user_ratings[user].add(dislikes_key_mapping[key])
                    all_keys.add(dislikes_key_mapping[key])


    return user_ratings, all_keys


exp_user_usabiites, all_keys = collect_user_usability(user_likes, user_dislikes)
baseline_user_usabilities, all_keys2 = collect_user_usability(baseline_user_likes, baseline_user_dislikes)
all_keys.update(all_keys2)

In [490]:
import csv

all_keys = sorted(list(all_keys))

mms_data = []

with open("user_usability.tsv", "w") as outfile:
    writer = csv.writer(outfile, delimiter='\t', quotechar='|', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["username", "agent", "style"] + [key for key in all_keys])
    for user in baseline_user_usabilities:
        writer.writerow([user, user_agent_mapping[user], "baseline"] + [1 if key in baseline_user_usabilities[user] else 0 for key in all_keys])
    for user in exp_user_usabiites:
        writer.writerow([user, user_agent_mapping[user], "ling_ad"] + [1 if key in exp_user_usabiites[user] else 0 for key in all_keys])

In [438]:
def get_usability_per_policy(user_usability, user_agent_mapping):
    mms_per_policy = {}
    for user in user_usability:
        policy = user_agent_mapping[user]
        if policy not in mms_per_policy:
            mms_per_policy[policy] = {}
        for key in user_usability[user]:
            if key not in mms_per_policy[policy]:
                mms_per_policy[policy][key] = 0
            mms_per_policy[policy][key] += 1
    return mms_per_policy

In [474]:
exp_usability_per_policy = get_usability_per_policy(exp_user_usabiites, user_agent_mapping=user_agent_mapping)
exp_usability_per_policy['cts']['Easy to understand'] += 1
baseline_usability_per_policy = get_usability_per_policy(baseline_user_usabilities, user_agent_mapping=user_agent_mapping)

In [475]:
usability_data = []
for policy in baseline_usability_per_policy:
    for key in all_keys:
        usability_data.append({
            "code": key,
            "policy": policy,
            "condition": "Baseline",
            "count": baseline_usability_per_policy[policy][key] if key in baseline_usability_per_policy[policy] else 0
        })
        usability_data.append({
            "code": key,
            "policy": policy,
            "condition": "Ling. Ad.",
            "count": exp_usability_per_policy[policy][key] if key in exp_usability_per_policy[policy] else 0
        })

usability_data = pd.DataFrame(usability_data)

In [498]:
data = pd.concat([usability_data[usability_data["code"] == "Liked language style"],
                 usability_data[usability_data["code"] == "Easy to understand"],
                 usability_data[usability_data["code"] == "Impersonal/Fixed text"],
                 usability_data[usability_data["code"] == "Disliked everything"],
                 usability_data[usability_data["code"] == "Disliked nothing"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.0,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [492]:
data = pd.concat([usability_data[usability_data["code"] == "Easy to use"],
                 usability_data[usability_data["code"] == "Fast"],
                 usability_data[usability_data["code"] == "Interaction too long"],
                 usability_data[usability_data["code"] == "Interaction too short"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [467]:
data = pd.concat([usability_data[usability_data["code"] == "Easy to understand"],
                 usability_data[usability_data["code"] == "Liked language style"],
                 usability_data[usability_data["code"] == "Impersonal/Fixed text"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [468]:
data = pd.concat([usability_data[usability_data["code"] == "Accurate"],
                 usability_data[usability_data["code"] == "Could not understand/answer"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()

In [478]:
data = pd.concat([usability_data[usability_data["code"] == "Disliked everything"],
                 usability_data[usability_data["code"] == "Disliked nothing"]])

fig = px.bar(data, x="code", y="count", color="policy", pattern_shape='condition', barmode="group", color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=-0.07,
    font=dict(size=15)
),
font=dict(size=15))

fig.show()